In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY"
assert os.getenv("LANGCHAIN_TRACING_V2") == "true", "LangSmith tracing not enabled"
assert os.getenv("LANGCHAIN_API_KEY"), "Missing LANGCHAIN_API_KEY"

print("✅ Environment variables loaded successfully!")

In [ ]:
import pandas as pd

df = pd.read_csv("data/journal_entries.csv")

print(f"📊 Loaded {len(df)} journal entries")
print(f"📋 Columns: {list(df.columns)}")
print("\n🔍 First few rows:")
df.head()

In [ ]:
from sox_copilot.evidence_agent import build_evidence_agent

agent = build_evidence_agent()

print("🤖 Agent built successfully!")

In [ ]:
import json 

control_id = "PAY-002"
period = "2024-07"
csv_path = "data/journal_entries.csv"
params = {
    "control_id": "PAY-002",
    "period": "2024-07",
    "csv_path": "data/journal_entries.csv",
}

print("🚀 Running end-to-end evidence agent...")
res = agent.invoke(params)

raw = res["output"]
report = json.loads(raw)
report



In [ ]:
from sox_copilot.tools import run_deterministic_check
facts = run_deterministic_check.invoke(params)
faithful = int(
    report["violations_found"]==facts["violations_found"] and
    set(report["violation_entries"])==set(facts["violation_entries"])
)
print({"faithfulness":faithful})

In [ ]:
from langsmith import Client
import pandas as pd, json
from sox_copilot.evidence_agent import build_evidence_agent
from sox_copilot.tools import run_deterministic_check

client = Client()
dataset = client.create_dataset("sox_eval_cases_3", description="PAY-002 audit controls")
df = pd.read_csv("data/eval_cases.csv")

for _, row in df.iterrows():
    client.create_example(inputs=row.to_dict(), dataset_id=dataset.id)

In [ ]:
def score_case(run, example):
    # Extract inputs from the example and outputs from the run
    inputs = example.inputs
    output = run.outputs
    
    # Run the deterministic check with the inputs
    facts = run_deterministic_check.invoke(inputs)
    
    # Compare the agent output with the facts
    faithful = int(
        output["violations_found"]==facts["violations_found"] and
        set(output["violation_entries"])==set(facts["violation_entries"])
    )
    return {"key": "faithfulness", "score": faithful}

# Create a wrapper function that accepts inputs
def run_agent(inputs):
    agent = build_evidence_agent()
    result = agent.invoke(inputs)
    return json.loads(result["output"])

client.evaluate(
    run_agent,
    data="sox_eval_cases_3",
    experiment_prefix="LangSmith Eval - PAY-002",
    max_concurrency=3,
    evaluators=[score_case],
)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.evaluation import EvaluatorType, load_evaluator

# load a prebuilt "criteria" evaluator (faithfulness / conciseness / helpfulness)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
evaluator = load_evaluator(EvaluatorType.CRITERIA, llm=llm, criteria="helpfulness")

def llm_grade(run, example):
    # Extract the narrative from the run outputs
    output = run.outputs
    text = output.get("narrative", "")
    
    # Extract the csv_path from the example inputs
    inputs = example.inputs
    
    grade = evaluator.evaluate_strings(
        input=inputs["csv_path"],  # or a short summary context
        prediction=text
    )
    # returns a dict like {"score":0.9,"explanation":"Narrative clearly explains violations."}
    return {"key": "narrative_quality", "score": grade["score"]}

In [ ]:
# Create a wrapper function that accepts inputs
def run_agent(inputs):
    agent = build_evidence_agent()
    result = agent.invoke(inputs)
    return json.loads(result["output"])

client.evaluate(
    run_agent,
    data="sox_eval_cases_3",
    experiment_prefix="LangSmith Eval - PAY-002 (with LLM Grader)",
    evaluators=[score_case, llm_grade],
)